# Import

In [1]:
import polars as pl
from constants import (
    DATA_FILE,
    PROCESSED_DATA_FOLDER,
    MODEL_FOLDER,
    DATA_TEST_FILE,
    DAY_CATEGORIES,
)
from helpers import clean_df
import json
import plotly.express as px
import lightgbm as lgb

In [2]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes: list[str] = json.load(f)
df_train = (
    pl.scan_parquet(DATA_FILE)
    .filter(pl.col("SubRouteID").is_in(target_routes))
    .pipe(clean_df)
)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops: dict[str, list[int]] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict: dict[str, str] = json.load(f)
ideal_tolerances = pl.read_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv").filter(
    pl.col("route").is_in(target_routes)
)

In [3]:
df = df_train.select(["Route", "Plate", "StopID", "Event", "Time"]).collect(
    engine="streaming"
)

In [4]:
round(df.estimated_size("gb"), 2)

4.79

- Should be able to fit in RAM in later functions

In [3]:
def asof_join_by_id(
    route_df: pl.DataFrame,
    depart_id: int,
    dest_id: int,
    tolerance: str,
) -> pl.DataFrame:
    depart_df = (
        route_df.filter(
            pl.col("Event") == "離站",
            pl.col("StopID") == depart_id,
        )
        .with_columns(pl.col("Time").alias("DepartureTime"))
        .sort("Time")
    )
    dest_df = (
        route_df.filter(
            pl.col("Event") == "進站",
            pl.col("StopID") == dest_id,
        )
        .with_columns(pl.col("Time").alias("ArrivalTime"))
        .sort("Time")
    )
    result = depart_df.join_asof(
        dest_df,
        on="Time",
        by="Plate",
        strategy="forward",
        tolerance=tolerance,
        check_sortedness=False,
    ).drop_nulls(subset=["ArrivalTime"])
    return result

In [4]:
def create_travel_time(df: pl.DataFrame, tolerances: pl.DataFrame) -> pl.DataFrame:
    results = []
    count = 0
    for target in tolerances.to_dicts():
        tol = target["tolerance"]
        df_target = df.filter(
            pl.col("Route") == target["route"],
            pl.col("StopID").is_in([target["depart_id"], target["dest_id"]]),
        )
        results.append(
            asof_join_by_id(df_target, target["depart_id"], target["dest_id"], tol)
        )
        count += 1
        if count % 500 == 0:
            print(f"{count} stop pairs have been processed.")
    return pl.concat(results) if results else pl.DataFrame()

In [21]:
df_with_travel_time = create_travel_time(df, ideal_tolerances)

500 stop pairs have been processed.
1000 stop pairs have been processed.
1500 stop pairs have been processed.
2000 stop pairs have been processed.
2500 stop pairs have been processed.
3000 stop pairs have been processed.
3500 stop pairs have been processed.
4000 stop pairs have been processed.
4500 stop pairs have been processed.
5000 stop pairs have been processed.
5500 stop pairs have been processed.
6000 stop pairs have been processed.


- It took 27 minutes 51 seconds to process 6367 stop pais joins.

In [28]:
df_with_travel_time.null_count()

Route,Plate,StopID,Event,Time,DepartureTime,Route_right,StopID_right,Event_right,ArrivalTime
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


- No null values

# Training Set Preparation

In [ ]:
# before doing anything else, save the output
df_with_travel_time.select(
    pl.col("StopID").alias("DepartureStop"),
    pl.col("StopID_right").alias("ArrivalStop"),
    pl.all(),
).with_columns(
    ((pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60).alias(
        "TravelDuration_min"
    ),
    (
        pl.col("DepartureTime").dt.hour().cast(pl.Int32) * 60
        + pl.col("DepartureTime").dt.minute().cast(pl.Int32)
    ).alias("MinutesFromMidnight"),
    pl.col("DepartureTime").dt.to_string("%a").cast(pl.Categorical).alias("DayOfWeek"),
).select(
    [
        "Route",
        "DepartureStop",
        "ArrivalStop",
        "MinutesFromMidnight",
        "DayOfWeek",
        "TravelDuration_min",
    ]
).write_parquet(PROCESSED_DATA_FOLDER / "train.parquet")

# Inspect the result quality

In [ ]:
train = pl.read_parquet(PROCESSED_DATA_FOLDER / "train.parquet")
px.histogram(train, x="TravelDuration_min")

In [37]:
train.filter(pl.col("TravelDuration_min") > 300).unique(
    subset=["Route", "DepartureStop", "ArrivalStop"]
).sort("Route")

Route,DepartureStop,ArrivalStop,MinutesFromMidnight,DayOfWeek,TravelDuration_min
str,i32,i32,i32,cat,f64
"""7500E1""",178344,303140,1132,"""Thu""",300.333333
"""7500M1""",178344,289798,895,"""Sun""",325.933333
"""7500Q1""",308538,289798,793,"""Sun""",321.783333
"""7513C2""",178510,303140,913,"""Sun""",302.266667
"""7513E2""",279706,303140,821,"""Sun""",320.6
"""7513G1""",289798,310064,559,"""Sat""",359.016667
"""7513G2""",279706,189257,706,"""Sun""",338.15


In [38]:
for s in [178344, 303140, 289798, 308538]:
    print(s, stops_dict[str(s)])

178344 新營站
303140 經國轉運站
289798 臺北轉運站
308538 永康轉運站


- The extremely high values all make sense.

In [39]:
train.filter(
    (300 > pl.col("TravelDuration_min")) & (pl.col("TravelDuration_min") > 250)
).unique(subset=["Route", "DepartureStop", "ArrivalStop"]).sort("Route")

Route,DepartureStop,ArrivalStop,MinutesFromMidnight,DayOfWeek,TravelDuration_min
str,i32,i32,i32,cat,f64
"""1610C2""",268481,301900,986,"""Sun""",273.833333
"""183301""",237066,276487,483,"""Fri""",286.266667
"""183302""",276487,267036,934,"""Sun""",255.3
"""183502""",269084,268304,880,"""Sun""",282.35
"""183602""",269084,140607,863,"""Sun""",256.233333
…,…,…,…,…,…
"""7513I2""",178344,189251,934,"""Thu""",255.3
"""7513J2""",178344,289798,1064,"""Sun""",253.883333
"""7513L1""",289798,178510,950,"""Sat""",250.566667


In [40]:
for s in [268481, 301900, 276487, 267036]:
    print(s, stops_dict[str(s)])

268481 楠梓站
301900 中壢服務區
276487 埔里站
267036 三重運動場


- All values make sense

# Test set preparation

In [5]:
df_test = (
    (
        pl.scan_parquet(DATA_TEST_FILE)
        .filter(pl.col("SubRouteID").is_in(target_routes))
        .pipe(clean_df)
    )
    .select(["Route", "Plate", "StopID", "Event", "Time"])
    .collect(engine="streaming")
)

In [6]:
df_test_with_travel_time = create_travel_time(df_test, ideal_tolerances)

500 stop pairs have been processed.
1000 stop pairs have been processed.
1500 stop pairs have been processed.
2000 stop pairs have been processed.
2500 stop pairs have been processed.
3000 stop pairs have been processed.
3500 stop pairs have been processed.
4000 stop pairs have been processed.
4500 stop pairs have been processed.
5000 stop pairs have been processed.
5500 stop pairs have been processed.
6000 stop pairs have been processed.


- It took 2 minutes 24 seconds to process 6367 stop pairs.

In [ ]:
# save the output
df_test_with_travel_time.select(
    pl.col("StopID").alias("DepartureStop"),
    pl.col("StopID_right").alias("ArrivalStop"),
    pl.all(),
).with_columns(
    ((pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60).alias(
        "TravelDuration_min"
    ),
    (
        pl.col("DepartureTime").dt.hour().cast(pl.Int32) * 60
        + pl.col("DepartureTime").dt.minute().cast(pl.Int32)
    ).alias("MinutesFromMidnight"),
    pl.col("DepartureTime").dt.to_string("%a").cast(pl.Categorical).alias("DayOfWeek"),
).select(
    [
        "Route",
        "DepartureStop",
        "ArrivalStop",
        "MinutesFromMidnight",
        "DayOfWeek",
        "TravelDuration_min",
    ]
).write_parquet(PROCESSED_DATA_FOLDER / "test.parquet")

In [26]:
test = pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")

In [ ]:
px.histogram(test, x="TravelDuration_min")

In [28]:
test.filter(pl.col("TravelDuration_min") > 300).unique(
    subset=["Route", "DepartureStop", "ArrivalStop"]
).sort("Route")

Route,DepartureStop,ArrivalStop,MinutesFromMidnight,DayOfWeek,TravelDuration_min
str,i32,i32,i32,cat,f64
"""7500M1""",178344,289798,957,"""Sun""",312.166667
"""7500Q1""",308538,289798,806,"""Sun""",332.45
"""7513C2""",178510,303140,889,"""Sun""",329.033333
"""7513E2""",279706,303140,806,"""Sun""",301.316667
"""7513G1""",289798,310064,469,"""Sat""",301.333333
"""7513G2""",279706,189257,931,"""Sun""",338.116667


- The test set and train set are comparable. 
# Convert to LightGBM.Dataset
## Naive Model (without target encoding)

In [25]:
# Cast "Route" as ENUM type to ensure identical categorical variable encoding for train and test set
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    routes: list[str] = json.load(f)
Routes_enum = pl.Enum(routes)
Days_enum = pl.Enum(DAY_CATEGORIES)

In [30]:
train = pl.read_parquet(PROCESSED_DATA_FOLDER / "train.parquet").with_columns(
    pl.col("Route").cast(Routes_enum).to_physical(),
    pl.col("DayOfWeek").cast(Days_enum).to_physical(),
)
test = pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet").with_columns(
    pl.col("Route").cast(Routes_enum).to_physical(),
    pl.col("DayOfWeek").cast(Days_enum).to_physical(),
)

In [27]:
cols = pl.read_parquet(PROCESSED_DATA_FOLDER / "train.parquet").columns
features = [c for c in cols if c != "TravelDuration_min"]

In [34]:
train_lgb = lgb.Dataset(
    data=train.drop("TravelDuration_min").to_numpy(),
    label=train["TravelDuration_min"].to_numpy(),
    feature_name=features,
    categorical_feature=[0, 1, 2, 4],
    free_raw_data=False,
)
test_lgb = lgb.Dataset(
    data=test.drop("TravelDuration_min").to_numpy(),
    label=test["TravelDuration_min"].to_numpy(),
    feature_name=features,
    categorical_feature=[0, 1, 2, 4],
    reference=train_lgb,
    free_raw_data=False,
)

In [35]:
train_lgb.save_binary(PROCESSED_DATA_FOLDER / "lightgbm_train.bin")
test_lgb.save_binary(PROCESSED_DATA_FOLDER / "lightgbm_test.bin")

## With target encoding
a bit messy due to how I originally handled the datasets haphazardly

In [10]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    routes: list[str] = json.load(f)
Routes_enum = pl.Enum(routes)
Days_enum = pl.Enum(DAY_CATEGORIES)

In [11]:
target_encoding = (
    pl.read_parquet(PROCESSED_DATA_FOLDER / "temp.parquet")
    .with_columns(
        (
            (pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60
        ).alias("TravelDuration")
    )
    .group_by(["Route", "DepartureStop", "ArrivalStop"])
    .agg(pl.mean("TravelDuration"))
)

In [ ]:
df_train = pl.read_parquet(PROCESSED_DATA_FOLDER / "temp.parquet")

In [ ]:
df_train_with_target_encoding = (
    df_train.join(
        target_encoding, on=["Route", "DepartureStop", "ArrivalStop"], how="left"
    )
    .with_columns(pl.col("TravelDuration").alias("MeanTravelTime"))
    .drop("TravelDuration")
)
df_test_with_target_encoding = (
    df_test_with_travel_time.select(
        pl.col("StopID").alias("DepartureStop"),
        pl.col("StopID_right").alias("ArrivalStop"),
        pl.all(),
    )
    .with_columns(
        (
            (pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60
        ).alias("TravelDuration_min"),
        (
            pl.col("DepartureTime").dt.hour().cast(pl.Int32) * 60
            + pl.col("DepartureTime").dt.minute().cast(pl.Int32)
        ).alias("MinutesFromMidnight"),
        pl.col("DepartureTime")
        .dt.to_string("%a")
        .cast(pl.Categorical)
        .alias("DayOfWeek"),
    )
    .select(
        [
            "Route",
            "DepartureStop",
            "ArrivalStop",
            "MinutesFromMidnight",
            "DayOfWeek",
            "TravelDuration_min",
        ]
    )
    .join(target_encoding, on=["Route", "DepartureStop", "ArrivalStop"], how="left")
    .with_columns(pl.col("TravelDuration").alias("MeanTravelTime"))
    .drop("TravelDuration")
)

In [15]:
df_test_with_target_encoding.select(
    [
        "Route",
        "MeanTravelTime",
        "MinutesFromMidnight",
        "DayOfWeek",
        "TravelDuration_min",
    ]
).write_parquet(PROCESSED_DATA_FOLDER / "test_target_encoding.parquet")

In [ ]:
df_train_with_target_encoding.with_columns(
    ((pl.col("ArrivalTime") - pl.col("DepartureTime")).dt.total_seconds() / 60).alias(
        "TravelDuration_min"
    ),
    (
        pl.col("DepartureTime").dt.hour().cast(pl.Int32) * 60
        + pl.col("DepartureTime").dt.minute().cast(pl.Int32)
    ).alias("MinutesFromMidnight"),
    pl.col("DepartureTime").dt.to_string("%a").cast(pl.Categorical).alias("DayOfWeek"),
).select(
    [
        "Route",
        "MeanTravelTime",
        "MinutesFromMidnight",
        "DayOfWeek",
        "TravelDuration_min",
    ]
).write_parquet(PROCESSED_DATA_FOLDER / "train_target_encoding.parquet")

In [19]:
train = pl.read_parquet(
    PROCESSED_DATA_FOLDER / "train_target_encoding.parquet"
).with_columns(
    pl.col("Route").cast(Routes_enum).to_physical(),
    pl.col("DayOfWeek").cast(Days_enum).to_physical(),
)
test = pl.read_parquet(
    PROCESSED_DATA_FOLDER / "test_target_encoding.parquet"
).with_columns(
    pl.col("Route").cast(Routes_enum).to_physical(),
    pl.col("DayOfWeek").cast(Days_enum).to_physical(),
)

In [20]:
cols = pl.read_parquet(PROCESSED_DATA_FOLDER / "train_target_encoding.parquet").columns
features = [c for c in cols if c != "TravelDuration_min"]

In [21]:
train_lgb = lgb.Dataset(
    data=train.drop("TravelDuration_min").to_numpy(),
    label=train["TravelDuration_min"].to_numpy(),
    feature_name=features,
    categorical_feature=[0, 3],
    free_raw_data=False,
)
test_lgb = lgb.Dataset(
    data=test.drop("TravelDuration_min").to_numpy(),
    label=test["TravelDuration_min"].to_numpy(),
    feature_name=features,
    categorical_feature=[0, 3],
    reference=train_lgb,
    free_raw_data=False,
)

In [ ]:
path = MODEL_FOLDER / "target_encoding_model"
train_lgb.save_binary(path / "lightgbm_train.bin")
test_lgb.save_binary(path / "lightgbm_test.bin")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Saving data to binary file C:\Code\Python\Projects\04-BusTime\model\target_encoding_model\lightgbm_test.bin
